# Geodes with EODAG

In this tutorial we will see how to use [EODAG](https://github.com/CS-SI/eodag) to search and download data on the [Geodes portal](https://geodes.cnes.fr).

*Disable some warnings related to geodes SSL verification disabled*

In [1]:
import requests
requests.packages.urllib3.disable_warnings() 

## 1. Credentials configuration

As explained on [Provider registration - geodes](https://eodag.readthedocs.io/en/stable/getting_started_guide/register.html#geodes), go to https://geodes-portal.cnes.fr, then login or create an account by clicking on `Log in` in the top-right corner. 

Once logged-in, create an API key in the user settings page, and use it as `apikey` in EODAG provider auth credentials.

- Update your EODAG user configuration file:
```yml
geodes:
    auth:
        credentials:
            apikey: your-api-key
```
- Or set it using environment variables:

In [2]:
import os
from getpass import getpass

os.environ["EODAG__GEODES__AUTH__CREDENTIALS__APIKEY"] = getpass("Input apikey: ")

Input apikey:  ········


## 2. Search

Let's instantiate EODAG `EODataAccessGateway`

In [3]:
from eodag import EODataAccessGateway

dag = EODataAccessGateway()

Then look for available product types on `geodes`

In [ ]:
dag.list_collections()

Search for October-2024 Sentinel 2 L1C data over South-West of France with less that 5% of cloud coverage

In [ ]:
search_bbox = [1, 43, 2, 44]
search_criteria = dict(
    provider="geodes",
    collection="S2_MSI_L1C",
    start="2024-10-01",
    end="2024-10-30",
    geom=search_bbox,
    cloudCover=5,
    count=True,
)

results = dag.search_all(**search_criteria)
results

View found products extents and search area on a map

In [7]:
import folium

# Create a map zoomed over the search area
fmap = folium.Map([43.5, 1.5], zoom_start=8)

# Create a layer that represents the search area in red
folium.Rectangle(
    bounds=[search_bbox[::-1][2:4], search_bbox[::-1][0:2]], 
    color="red", 
    tooltip="Search extent"
).add_to(fmap)

# Create a layer that maps the products found
folium.GeoJson(
    data=results, tooltip=folium.GeoJsonTooltip(fields=["title"])
).add_to(fmap)

fmap

## 3. Download products to `/work/scratch/data/{USER}"` directory

In [ ]:
paths = dag.download_all(results, output_dir=f"/work/scratch/data/{os.environ['JUPYTERHUB_USER']}")
paths

In [ ]:
!tree {paths[0]}